In [3]:
"""
- 输入：把 30x30 迷宫展平成长度 900 的一维向量（每个格子映射到 0~4）
- 网络：900 -> 4096 -> 512 -> 4（ReLU）
- 训练：用 train_data.csv + train_answer.csv 做回归（MSE），Adam，小批量（PyTorch）
- 输出：对 test_data.csv 预测，写 result.csv（每行 4 个实数）

用法：
    python baseline.py train_data.csv train_answer.csv test_data.csv result.csv
"""

import sys
from pathlib import Path
import pandas as pd
import numpy as np
import torch
import torch.nn as nn
from torch.utils.data import DataLoader, TensorDataset
import zipfile
import os
import random
from collections import deque

seed = 42

random.seed(seed)                  # Python built-in random
np.random.seed(seed)               # NumPy
torch.manual_seed(seed)            # PyTorch (CPU)
torch.cuda.manual_seed(seed)       # PyTorch (single GPU)
torch.cuda.manual_seed_all(seed)   # PyTorch (all GPUs)

# Ensures deterministic behavior
torch.backends.cudnn.deterministic = True
torch.backends.cudnn.benchmark = False

N = 30
D = N * N

# ===== 数据读取与编码 =====

def encode_lines(path: str) -> np.ndarray:
    # 将每行 900 字符编码成 0~4 的向量：
    # '.'->0, '# '->1, '?'->2, 'S'->3, 'T'->4
    char_id = {".": 0, "#": 1, "?": 2, "S": 3, "T": 4}
    with open(path, "r", encoding="utf-8-sig") as f:
        xs = [[char_id[c] for c in line.strip()] for line in f if line.strip()]
    return np.asarray(xs, dtype=np.float32)


def read_y(path: str) -> np.ndarray:
    # 读取 4 列标签（整数），训练时当作 float
    return np.loadtxt(path, delimiter=",", dtype=np.float32, encoding="utf-8-sig")


# ===== 结果写出 =====

def write_result(path: Path, pred: np.ndarray) -> None:
    np.savetxt(path, pred, delimiter=",", fmt="%.6f")


# ===== 模型训练 =====

def train_model(train_x_path: str, train_y_path: str, epochs: int) -> nn.Module:
    # 输入缩放到约 [0,1]，标签除以 900 后回归，预测时再乘回原量纲。
    x_train = torch.from_numpy(encode_lines(train_x_path) / 4.0)
    y_train = torch.from_numpy(read_y(train_y_path) / 900.0)

    torch.manual_seed(0)
    model = nn.Sequential(
        nn.Linear(D, 4096), nn.ReLU(),
        nn.Linear(4096, 512), nn.ReLU(),
        nn.Linear(512, 4),
    )
    loader = DataLoader(TensorDataset(x_train, y_train), batch_size=64, shuffle=True)
    opt = torch.optim.Adam(model.parameters(), lr=1e-3)

    model.train()
    for ep in range(1, epochs + 1):
        total = 0.0
        for xb, yb in loader:
            pred = model(xb)
            loss = nn.functional.mse_loss(pred, yb)
            opt.zero_grad()
            loss.backward()
            opt.step()
            total += float(loss.item()) * xb.shape[0]
        print(f"epoch {ep}/{epochs}  mse={total / len(x_train):.6f}")
    return model


# ===== 预测 =====

def predict(model: nn.Module, test_x_path: str) -> np.ndarray:
    x_test = torch.from_numpy(encode_lines(test_x_path) / 4.0)
    model.eval()
    mp = []
    with open(test_x_path, "r", encoding="utf-8-sig") as f:
        for line in f:
            if line.strip():
                mp.append(line.strip())
    
    def get(idx, i, j):
        return mp[idx][i * 30 + j]

    def bfs(start, target):
        queue = deque([start])
        visited = set()
        visited.add(start)
        distance = 0
        
        while queue:
            for _ in range(len(queue)):
                x, y = queue.popleft()
                if (x, y) == target:
                    return distance
                for dx, dy in [(-1, 0), (1, 0), (0, -1), (0, 1)]:
                    nx, ny = x + dx, y + dy
                    if 0 <= nx < 30 and 0 <= ny < 30 and (nx, ny) not in visited:
                        if get(i, nx, ny) in ['S', 'T', '.', '?']:
                            visited.add((nx, ny))
                            queue.append((nx, ny))
            distance += 1
        return float('inf')  # 如果无法到达目标

    results = []

    for i in range(len(mp)):
        min_c1 = float('inf')  # 障碍单元格总数的最小值
        max_c1 = float('-inf')  # 障碍单元格总数的最大值
        min_c2 = float('inf')  # 可到达的空地单元格总数的最小值
        max_c2 = float('-inf')  # 可到达的空地单元格总数的最大值
        min_components = float('inf')  # 四连通块数量的最小值
        max_components = float('-inf')  # 四连通块数量的最大值
        min_shortest_path_length = float('inf')  # 最短路径长度的最小值
        max_shortest_path_length = float('-inf')  # 最短路径长度的最大值

        for treat_question_mark_as in ['.', '#']:
            c1 = 0  # 障碍单元格总数
            c2 = 0  # 可到达的空地单元格总数
            S = None
            T = None
            visited = set()
            components = 0  # 四连通块数量
            
            # 计算障碍和可通行单元格
            for x in range(30):
                for y in range(30):
                    cell = get(i, x, y)
                    if cell == '#':
                        c1 += 1
                    elif cell in ['S', 'T', '.']:
                        if cell == 'S':
                            S = (x, y)
                        elif cell == 'T':
                            T = (x, y)
                        if (x, y) not in visited:
                            # 进行 BFS 计算四连通块
                            components += 1
                            queue = deque([(x, y)])
                            while queue:
                                cx, cy = queue.popleft()
                                visited.add((cx, cy))
                                for dx, dy in [(-1, 0), (1, 0), (0, -1), (0, 1)]:
                                    nx, ny = cx + dx, cy + dy
                                    if 0 <= nx < 30 and 0 <= ny < 30 and (nx, ny) not in visited:
                                        if get(i, nx, ny) in ['S', 'T', '.', '?']:
                                            visited.add((nx, ny))
                                            queue.append((nx, ny))
                    elif cell == '?':
                        if treat_question_mark_as == '.':
                            c2 += 1  # 将 ? 视为可通行
                        else:
                            c1 += 1  # 将 ? 视为障碍

            # 计算从 S 到 T 的最短路径长度
            if S and T:
                shortest_path_length = bfs(S, T)
            else:
                shortest_path_length = float('inf')  # 如果没有 S 或 T
            
            # 更新最小值和最大值
            min_c1 = min(min_c1, c1)
            max_c1 = max(max_c1, c1)
            min_c2 = min(min_c2, c2)
            max_c2 = max(max_c2, c2)
            min_components = min(min_components, components)
            max_components = max(max_components, components)
            min_shortest_path_length = min(min_shortest_path_length, shortest_path_length)
            max_shortest_path_length = max(max_shortest_path_length, shortest_path_length)

        # 计算平均值
        avg_c1 = (min_c1 + max_c1) / 2
        avg_c2 = (min_c2 + max_c2) / 2
        avg_components = (min_components + max_components) / 2
        avg_shortest_path_length = (min_shortest_path_length + max_shortest_path_length) / 2

        # 将结果添加到结果列表
        results.append([avg_c1, avg_c2, avg_components, avg_shortest_path_length])

    return np.array(results)


# ===== 主流程 =====


TRAIN_PATH = "/bohr/train-abk9/v1/"  # 训练集路径


# 训练集
train_x_path = TRAIN_PATH + "train_data.csv"
train_y_path = TRAIN_PATH + "train_answer.csv"



out_path = Path("result.csv")
epochs = 3

model = train_model(train_x_path, train_y_path, epochs)




# 保存模型权重和必要的缩放信息，便于复现预测。
torch.save(
    {
        "state_dict": model.state_dict(),
        "epochs": epochs,
        "architecture": "900-4096-512-4",
        "input_scale": 4.0,
        "output_scale": 900.0,
    },
    out_path.with_name("model.pt"),
)


In [4]:
if os.environ.get("DATA_PATH"):
    DATA_PATH = os.environ.get("DATA_PATH") + "/"  # 测试集路径
else:
    DATA_PATH = "/bohr/mazeval-7zx2/v1/"  # 本地测试回退

# 测试集
testA_path = DATA_PATH + "val_data.csv"
testB_path = DATA_PATH + "test_data.csv"

#分别预测
pred_A = predict(model, testA_path)
pred_B = predict(model, testB_path)

#合并预测结果

submissionA = pd.DataFrame(pred_A)
submissionA.to_csv("./submission_val.csv", index=False, header=False)

submissionB = pd.DataFrame(pred_B)
submissionB.to_csv("./submission_test.csv", index=False, header=False)

files_to_zip = ['./submission_val.csv', './submission_test.csv']
zip_filename = 'submission.zip'

with zipfile.ZipFile(zip_filename, 'w') as zipf:
    for file in files_to_zip:
        zipf.write(file, os.path.basename(file))

print(f'{zip_filename} is created succefully!')